# Securing the Digital Mine: Google Colab GPU Training
## Run this notebook on Google Colab with a T4 GPU for best results.
## Runtime > Change runtime type > T4 GPU
## Estimated training time: 15-25 minutes on T4 GPU

This notebook is fully self-contained. It will:
1. Verify GPU availability
2. Clone the repository and install dependencies
3. Download the NSL-KDD dataset automatically
4. Run preprocessing
5. Run BWOA feature selection
6. Train CNN-LSTM v4 (strengthened architecture) on GPU
7. Evaluate on KDDTest+
8. Quantize and benchmark
9. Download all output artifacts
10. (Optional) Push to GitHub

## Cell 1: Runtime Check and GPU Verification

In [ ]:
import subprocess, sys

gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if 'failed' in gpu_info.stdout or gpu_info.returncode != 0:
    print('WARNING: No GPU detected.')
    print('Go to Runtime > Change runtime type > T4 GPU and re-run.')
else:
    lines = [l for l in gpu_info.stdout.split('\n') if l.strip()]
    print('GPU detected:', lines[8] if len(lines) > 8 else lines[-1])
    print('Full nvidia-smi output:')
    print(gpu_info.stdout[:600])

# Also verify TF sees the GPU
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'\nTensorFlow version: {tf.__version__}')
print(f'GPUs available: {gpus}')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print('Memory growth enabled on GPU 0.')
else:
    print('WARNING: TF cannot see a GPU. Training will use CPU only (much slower).')

## Cell 2: Clone Repository and Install Dependencies

In [ ]:
%%bash
set -e

# Clone repository if not already present
if [ ! -d 'unesco-project' ]; then
    git clone https://github.com/mhiskall282/unesco-project.git
    echo 'Repository cloned.'
else
    cd unesco-project && git pull origin main && cd ..
    echo 'Repository updated.'
fi

cd unesco-project

# Install GPU-pinned dependencies from colab_requirements.txt
pip install -q tensorflow==2.15.0 scikit-learn==1.4.0 pandas==2.1.4 numpy==1.26.3 \
    matplotlib==3.8.2 seaborn==0.13.1 pyyaml==6.0.1 tqdm==4.66.1 joblib==1.3.2

# tflite-runtime is optional; fall back to tf.lite if unavailable
pip install -q tflite-runtime || echo 'tflite-runtime not available on this Python, using tf.lite fallback'

echo 'All dependencies installed.'

## Cell 3: Download NSL-KDD Dataset

In [ ]:
%%bash
set -e
cd unesco-project
mkdir -p data/raw

if [ ! -f data/raw/KDDTrain+.txt ]; then
    echo 'Downloading KDDTrain+.txt...'
    wget -q 'https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain+.txt' \
         -O data/raw/KDDTrain+.txt
fi

if [ ! -f data/raw/KDDTest+.txt ]; then
    echo 'Downloading KDDTest+.txt...'
    wget -q 'https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest+.txt' \
         -O data/raw/KDDTest+.txt
fi

echo 'Line counts:'
wc -l data/raw/KDDTrain+.txt data/raw/KDDTest+.txt
echo 'Download complete.'

## Cell 4: Run Full Preprocessing Pipeline

In [ ]:
import sys, os
sys.path.insert(0, '/content/unesco-project')
os.chdir('/content/unesco-project')

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import pickle

# NSL-KDD column names (41 features + label + difficulty)
COL_NAMES = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty'
]

ATTACK_MAP = {
    'normal': 'Normal',
    # DoS
    'neptune': 'DoS', 'back': 'DoS', 'land': 'DoS', 'pod': 'DoS',
    'smurf': 'DoS', 'teardrop': 'DoS', 'mailbomb': 'DoS', 'apache2': 'DoS',
    'processtable': 'DoS', 'udpstorm': 'DoS',
    # Probe
    'ipsweep': 'Probe', 'nmap': 'Probe', 'portsweep': 'Probe', 'satan': 'Probe',
    'mscan': 'Probe', 'saint': 'Probe',
    # R2L
    'ftp_write': 'R2L', 'guess_passwd': 'R2L', 'imap': 'R2L', 'multihop': 'R2L',
    'phf': 'R2L', 'spy': 'R2L', 'warezclient': 'R2L', 'warezmaster': 'R2L',
    'sendmail': 'R2L', 'named': 'R2L', 'snmpgetattack': 'R2L', 'snmpguess': 'R2L',
    'xlock': 'R2L', 'xsnoop': 'R2L', 'worm': 'R2L',
    # U2R
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R', 'rootkit': 'U2R',
    'httptunnel': 'U2R', 'ps': 'U2R', 'sqlattack': 'U2R', 'xterm': 'U2R',
}

CATEGORICAL_FEATURES = ['protocol_type', 'service', 'flag']
CLASS_LABELS = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']

def load_nslkdd(path):
    df = pd.read_csv(path, names=COL_NAMES, header=None)
    df = df.drop('difficulty', axis=1)
    df['label'] = df['label'].str.strip('.').map(ATTACK_MAP)
    df = df.dropna(subset=['label'])
    return df

print('Loading NSL-KDD...')
df_train = load_nslkdd('data/raw/KDDTrain+.txt')
df_test  = load_nslkdd('data/raw/KDDTest+.txt')
print(f'Train: {len(df_train)} rows | Test: {len(df_test)} rows')
print('Class distribution (train):')
print(df_train['label'].value_counts())

# Encode categorical features
encoders = {}
for col in CATEGORICAL_FEATURES:
    le = LabelEncoder()
    le.fit(pd.concat([df_train[col], df_test[col]]))
    df_train[col] = le.transform(df_train[col])
    df_test[col]  = le.transform(df_test[col].map(lambda x: x if x in le.classes_ else le.classes_[0]))
    encoders[col] = le

# Encode labels
label_le = LabelEncoder()
label_le.fit(CLASS_LABELS)
y_train = label_le.transform(df_train['label'])
y_test  = label_le.transform(df_test['label'])

feature_cols = [c for c in COL_NAMES[:-2]]
X_train = df_train[feature_cols].values.astype(np.float32)
X_test  = df_test[feature_cols].values.astype(np.float32)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Save
os.makedirs('data/processed', exist_ok=True)
os.makedirs('data/features', exist_ok=True)

np.save('data/processed/X_train.npy', X_train)
np.save('data/processed/X_test.npy',  X_test)
np.save('data/processed/y_train.npy', y_train)
np.save('data/processed/y_test.npy',  y_test)

with open('data/processed/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Preprocessing complete. Saved to data/processed/')
print(f'X_train shape: {X_train.shape} | X_test shape: {X_test.shape}')

## Cell 5: BWOA Feature Selection (GPU-Friendly RandomForest)

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from tqdm.notebook import tqdm
import sys, os

sys.path.insert(0, '/content/unesco-project')
os.chdir('/content/unesco-project')

X_train = np.load('data/processed/X_train.npy')
y_train = np.load('data/processed/y_train.npy')
X_test  = np.load('data/processed/X_test.npy')
y_test  = np.load('data/processed/y_test.npy')

# Use a stratified 3000-sample subset for BWOA fitness evaluation (faster)
from sklearn.model_selection import StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_tr, idx_val = next(sss.split(X_train[:3000], y_train[:3000]))
X_bwoa_tr, y_bwoa_tr = X_train[:3000][idx_tr], y_train[:3000][idx_tr]
X_bwoa_val, y_bwoa_val = X_train[:3000][idx_val], y_train[:3000][idx_val]

MIN_ACCURACY = 0.75

def fitness_fn(mask, X_tr, y_tr, X_val, y_val, alpha=0.3):
    """BWOA fitness: weighted sum of (1-accuracy) and feature ratio."""
    selected = np.where(mask == 1)[0]
    if len(selected) == 0:
        return 1.0  # invalid
    clf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    scores = cross_val_score(clf, X_tr[:, selected], y_tr, cv=3, scoring='accuracy')
    acc = scores.mean()
    if acc < MIN_ACCURACY:
        return 1.0  # reject
    feature_ratio = len(selected) / X_tr.shape[1]
    return alpha * (1.0 - acc) + (1.0 - alpha) * feature_ratio

from src.optimization.bwoa import BinaryWhaleOptimizer

N_AGENTS   = 30
MAX_ITER   = 100
N_FEATURES = X_train.shape[1]

print(f'Running BWOA: n_agents={N_AGENTS}, max_iter={MAX_ITER}, n_features={N_FEATURES}')

optimizer = BinaryWhaleOptimizer(
    n_agents=N_AGENTS,
    n_features=N_FEATURES,
    max_iter=MAX_ITER,
    fitness_fn=fitness_fn,
    minimum_features=10,
    alpha_start=0.5,
    alpha_end=0.3,
    alpha_decay_iters=50,
    diversity_threshold=0.1,
)

best_mask, fitness_history = optimizer.optimize(
    X_bwoa_tr, y_bwoa_tr, X_bwoa_val, y_bwoa_val, patience=15
)

selected_indices = np.where(best_mask == 1)[0]
print(f'\nBWOA complete. Selected {len(selected_indices)} features out of {N_FEATURES}.')
print(f'Selected feature indices: {selected_indices.tolist()}')

np.save('data/features/nslkdd_bwoa_mask_v3.npy', best_mask)
print('Feature mask saved to data/features/nslkdd_bwoa_mask_v3.npy')

## Cell 6: Mount Google Drive and Train CNN-LSTM v4 on GPU

In [ ]:
# Mount Google Drive for persistent checkpoint storage
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/DigitalMine'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive checkpoint directory: {DRIVE_DIR}')

In [ ]:
import numpy as np
import tensorflow as tf
import sys, os

sys.path.insert(0, '/content/unesco-project')
os.chdir('/content/unesco-project')

from src.models.cnn_lstm import build_cnn_lstm_v4
from tensorflow.keras.utils import to_categorical

CLASS_LABELS = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
N_CLASSES = len(CLASS_LABELS)

# Load preprocessed data
X_train_full = np.load('data/processed/X_train.npy')
y_train_full = np.load('data/processed/y_train.npy')
X_test       = np.load('data/processed/X_test.npy')
y_test       = np.load('data/processed/y_test.npy')

# Load BWOA mask and select features
mask = np.load('data/features/nslkdd_bwoa_mask_v3.npy')
sel  = np.where(mask == 1)[0]
print(f'Using {len(sel)} BWOA-selected features.')

X_tr = X_train_full[:, sel].reshape(-1, len(sel), 1).astype(np.float32)
X_te = X_test[:, sel].reshape(-1, len(sel), 1).astype(np.float32)

# Balanced class weights to handle NSL-KDD imbalance
from sklearn.utils.class_weight import compute_class_weight
class_weights_arr = compute_class_weight('balanced', classes=np.arange(N_CLASSES), y=y_train_full)
class_weight_dict = {i: w for i, w in enumerate(class_weights_arr)}
print('Class weights:', class_weight_dict)

y_tr_cat = to_categorical(y_train_full, num_classes=N_CLASSES)
y_te_cat = to_categorical(y_test, num_classes=N_CLASSES)

# Build CNN-LSTM v4
input_shape = (X_tr.shape[1], 1)
model = build_cnn_lstm_v4(
    input_shape=input_shape,
    n_classes=N_CLASSES,
    filters=64,
    lstm_units=256,
    dropout_rate=0.3,
    label_smoothing=0.1,
    l2_strength=1e-4,
    learning_rate=1e-3,
)
model.summary()

# Cosine annealing learning rate schedule
EPOCHS     = 100
BATCH_SIZE = 256
STEPS_PER_EPOCH = len(X_tr) // BATCH_SIZE

cosine_lr = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3,
    decay_steps=EPOCHS * STEPS_PER_EPOCH,
    alpha=1e-5,
)
model.optimizer.learning_rate = cosine_lr

# Callbacks
DRIVE_DIR = '/content/drive/MyDrive/DigitalMine'
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=15, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=f'{DRIVE_DIR}/cnn_lstm_v4_best.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.TensorBoard(log_dir='/content/tb_logs', histogram_freq=0),
]

print(f'Training CNN-LSTM v4 for up to {EPOCHS} epochs (batch={BATCH_SIZE})...')
history = model.fit(
    X_tr, y_tr_cat,
    validation_data=(X_te, y_te_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1,
)

# Save local copy
os.makedirs('models', exist_ok=True)
model.save('models/cnn_lstm_v4_colab.keras')
print('Model saved to models/cnn_lstm_v4_colab.keras')

## Cell 7: Evaluate on KDDTest+ (Classification Report + Plots)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize
import os

os.makedirs('figures', exist_ok=True)

CLASS_LABELS = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']

# Predictions
y_prob = model.predict(X_te, batch_size=256, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
y_true = y_test

# Classification report
print('=== Classification Report (KDDTest+) ===')
print(classification_report(y_true, y_pred, target_names=CLASS_LABELS, digits=4))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS, ax=ax)
ax.set_title('CNN-LSTM v4 Confusion Matrix (KDDTest+)')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.tight_layout()
plt.savefig('figures/confusion_matrix_v4.png', dpi=150)
plt.show()
print('Confusion matrix saved to figures/confusion_matrix_v4.png')

# Training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Accuracy')
axes[0].legend()
axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Loss')
axes[1].legend()
plt.tight_layout()
plt.savefig('figures/training_history_v4.png', dpi=150)
plt.show()
print('Training history saved to figures/training_history_v4.png')

# ROC curves (One-vs-Rest)
y_bin = label_binarize(y_true, classes=list(range(len(CLASS_LABELS))))
fig, ax = plt.subplots(figsize=(8, 6))
for i, cls in enumerate(CLASS_LABELS):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{cls} (AUC={roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--')
ax.set_title('ROC Curves (OvR) - CNN-LSTM v4')
ax.set_xlabel('FPR')
ax.set_ylabel('TPR')
ax.legend()
plt.tight_layout()
plt.savefig('figures/roc_curves_v4.png', dpi=150)
plt.show()
print('ROC curves saved to figures/roc_curves_v4.png')

## Cell 8: Float16 Quantization and Latency Benchmark

In [ ]:
import tensorflow as tf
import numpy as np
import time, os

os.makedirs('models', exist_ok=True)

print('Converting to TFLite Float16...')
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

TFLITE_PATH = 'models/cnn_lstm_v4_colab_quantized.tflite'
with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

size_mb = os.path.getsize(TFLITE_PATH) / 1e6
print(f'Quantized model size: {size_mb:.4f} MB')

# Load TFLite interpreter
try:
    import tflite_runtime.interpreter as tflite
    interpreter = tflite.Interpreter(model_path=TFLITE_PATH)
except ImportError:
    interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)

interpreter.allocate_tensors()
input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Warm-up runs
sample = X_te[:1].astype(np.float32)
for _ in range(10):
    interpreter.set_tensor(input_details[0]['index'], sample)
    interpreter.invoke()

# Benchmark: 1000 inference runs
N_RUNS = 1000
latencies = []
for _ in range(N_RUNS):
    interpreter.set_tensor(input_details[0]['index'], sample)
    t0 = time.perf_counter()
    interpreter.invoke()
    latencies.append((time.perf_counter() - t0) * 1000)

latencies = np.array(latencies)
print(f'\n=== TFLite Float16 Inference Benchmark ({N_RUNS} runs) ===')
print(f'Model size:     {size_mb:.4f} MB')
print(f'Latency mean:   {latencies.mean():.4f} ms')
print(f'Latency std:    {latencies.std():.4f} ms')
print(f'Latency P50:    {np.percentile(latencies, 50):.4f} ms')
print(f'Latency P95:    {np.percentile(latencies, 95):.4f} ms')
print(f'Latency P99:    {np.percentile(latencies, 99):.4f} ms')
print(f'Latency max:    {latencies.max():.4f} ms')

## Cell 9: Download Output Artifacts

In [ ]:
from google.colab import files
import os

OUTPUT_FILES = [
    'models/cnn_lstm_v4_colab.keras',
    'models/cnn_lstm_v4_colab_quantized.tflite',
    'data/features/nslkdd_bwoa_mask_v3.npy',
    'data/processed/scaler.pkl',
    'figures/confusion_matrix_v4.png',
    'figures/training_history_v4.png',
    'figures/roc_curves_v4.png',
]

for filepath in OUTPUT_FILES:
    full_path = f'/content/unesco-project/{filepath}'
    if os.path.exists(full_path):
        print(f'Downloading {filepath}...')
        files.download(full_path)
    else:
        print(f'MISSING: {filepath} -- run previous cells first.')

print('\nDownload complete. Place the .tflite and .npy files in your local repo:')
print('  models/cnn_lstm_v4_colab_quantized.tflite')
print('  data/features/nslkdd_bwoa_mask_v3.npy')
print('  data/processed/scaler.pkl')
print('Then scp them to your Raspberry Pi using Section 8 of docs/raspberry_pi_deployment.md')

## Cell 10: Upload to GitHub (Optional)

In [ ]:
# This cell is OPTIONAL. Only run it if you want to push trained artifacts back to GitHub.
# Note: model files are gitignored by default (.gitignore). You will need to
# git add -f models/*.tflite to force-add them, or upload them to a GitHub Release asset.
#
# Step 1: Configure your GitHub credentials
# !git config --global user.email 'you@example.com'
# !git config --global user.name 'Your Name'
#
# Step 2: Add and commit the trained artifacts
# %%bash
# cd /content/unesco-project
# git add -f models/cnn_lstm_v4_colab_quantized.tflite
# git add -f data/features/nslkdd_bwoa_mask_v3.npy
# git add -f data/processed/scaler.pkl
# git commit -m 'feat: add v4 quantized model and BWOA mask from Colab training'
#
# Step 3: Push using a Personal Access Token (PAT)
# (Colab cannot use SSH keys interactively)
# Replace <PAT> and <USERNAME> with your actual values:
# git remote set-url origin https://<USERNAME>:<PAT>@github.com/mhiskall282/unesco-project.git
# git push origin main
#
# Alternatively: upload the .tflite file as a GitHub Release asset at:
# https://github.com/mhiskall282/unesco-project/releases

print('Cell 10: GitHub upload steps are provided as comments above.')
print('Edit the commented-out commands and uncomment to use them.')
print('Alternatively, download artifacts in Cell 9 and push from your local machine.')